# 🧠 EEG-to-Speech Architecture Tutorial

This notebook walks through the **full forward pass** of our EEG-to-Speech model.

We will:
1. Load a raw EEG `.npy` file and inspect its dimensions
2. Visualize the complete architecture
3. Pass the EEG through **EEGModule → SpeechDecoder** (inference mode)
4. Print the shape at **every intermediate stage**
5. Show EEG metadata (channels, frames, frequency)
6. Show output audio metadata (frames, duration, sample rate)

---
## 0. Setup & Imports

In [ ]:
import os
import sys
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import IPython.display as ipd

%matplotlib inline

# ── Project paths ──
PROJECT_ROOT = os.path.abspath('.')  # Run this notebook from the project root
sys.path.insert(0, PROJECT_ROOT)

EEG_FILE = "/media/hdd1/amkapoor/N400/N400Stimset_manuscriptdata/N400Stimset_manuscriptdata/N400_epoched/sub-01/sub-01-_-NPC_aisle.wav.npy"
CHECKPOINT_DIR = "/media/hdd3/amkapoor/logs/subject_disc_3"
CONFIG_PATH = os.path.join(CHECKPOINT_DIR, "configs.json")

# Fallback to project config if not found at checkpoint dir
if not os.path.exists(CONFIG_PATH):
    CONFIG_PATH = os.path.join(PROJECT_ROOT, "configs", "configs.json")
    print(f"[INFO] Config not found in checkpoint dir, falling back to: {CONFIG_PATH}")

EEG_SAMPLE_RATE = 256  # Hz

print(f"Project root : {PROJECT_ROOT}")
print(f"EEG file     : {EEG_FILE}")
print(f"Checkpoint   : {CHECKPOINT_DIR}")
print(f"Config       : {CONFIG_PATH}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

---
## 2. Load Raw EEG Data

We load the `.npy` file and inspect its original dimensions, sample rate, and duration.

In [ ]:
eeg_data = np.load(EEG_FILE)

print(f"📂 File          : {EEG_FILE}")
print(f"📐 NumPy shape   : {eeg_data.shape}")
print(f"📊 Data type     : {eeg_data.dtype}")
print(f"📏 Min / Max     : {eeg_data.min():.4f} / {eeg_data.max():.4f}")

# Handle 2D or 3D arrays
if eeg_data.ndim == 2:
    n_channels, n_frames = eeg_data.shape
elif eeg_data.ndim == 3:
    print(f"⚠️  3D array detected (shape {eeg_data.shape}). Using first slice, last 2 dims.")
    eeg_data = eeg_data[0] if eeg_data.shape[0] > 1 else eeg_data.squeeze(0)
    n_channels, n_frames = eeg_data.shape
else:
    raise ValueError(f"Unexpected EEG shape: {eeg_data.shape}")

duration_sec = n_frames / EEG_SAMPLE_RATE

print(f"\n── EEG Signal Properties ──")
print(f"🧠 Channels      : {n_channels}")
print(f"🔢 Frames (T)    : {n_frames}")
print(f"⏱️  Sample Rate   : {EEG_SAMPLE_RATE} Hz")
print(f"⏳ Duration       : {duration_sec:.3f} seconds")

---
## 3. Load Model Architecture & Checkpoint

In [ ]:
import utils
from EEGModule import EEGModule
from models import SpeechDecoder

hps = utils.get_hparams_from_file(CONFIG_PATH)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"📄 Config loaded from: {CONFIG_PATH}")
print(f"🖥️  Device: {device}")

print(f"\n── Model Config ──")
print(f"  inter_channels     : {hps.model.inter_channels}")
print(f"  hidden_channels    : {hps.model.hidden_channels}")
print(f"  filter_channels    : {hps.model.filter_channels}")
print(f"  n_heads            : {hps.model.n_heads}")
print(f"  n_layers           : {hps.model.n_layers}")
print(f"  upsample_rates     : {hps.model.upsample_rates}  (product = {math.prod(hps.model.upsample_rates)})")
print(f"  EEG n_layers_cnn   : {hps.model.eeg_module.n_layers_cnn}")
print(f"  EEG use_s4         : {hps.model.eeg_module.use_s4}")
print(f"  EEG n_layers_s4    : {hps.model.eeg_module.n_layers_s4}")
print(f"  EEG in_channels    : {hps.model.eeg_module.in_channels}")
print(f"  Audio sample rate  : {hps.data.sampling_rate}")
print(f"  Audio hop_length   : {hps.data.hop_length}")

In [ ]:
# Build models
eeg_module = EEGModule(
    n_layers_cnn=hps.model.eeg_module.n_layers_cnn,
    use_s4=hps.model.eeg_module.use_s4,
    n_layers_s4=hps.model.eeg_module.n_layers_s4,
    embedding_size=hps.model.inter_channels,
    is_mask=False,
    in_channels=hps.model.eeg_module.in_channels,
    num_subjects=hps.model.num_subjects if hasattr(hps.model, 'num_subjects') else 25,
    device=device
).to(device)

net_g = SpeechDecoder(
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model
).to(device)

print("✅ Models built successfully")
print(f"   EEGModule params  : {sum(p.numel() for p in eeg_module.parameters()):,}")
print(f"   SpeechDecoder params: {sum(p.numel() for p in net_g.parameters()):,}")

In [ ]:
# Find and load the latest checkpoint
ckpt_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("E_") and f.endswith(".pth")]
iterations = []
for f in ckpt_files:
    try:
        idx = int(f.replace("E_", "").replace(".pth", ""))
        iterations.append(idx)
    except ValueError:
        pass
latest_iter = max(iterations)

e_path = os.path.join(CHECKPOINT_DIR, f"E_{latest_iter}.pth")
g_path = os.path.join(CHECKPOINT_DIR, f"G_{latest_iter}.pth")

print(f"📦 Loading EEG checkpoint   : {e_path}")
print(f"📦 Loading Speech checkpoint : {g_path}")

utils.load_checkpoint(e_path, eeg_module, None)
utils.load_checkpoint(g_path, net_g, None)

eeg_module.eval()
net_g.eval()

print(f"\n✅ Checkpoints loaded (iteration {latest_iter})")

---
## 4. Forward Pass — Tracing Dimensions Through EEG Module

We manually trace through each sub-module of the `EEGModule` and print the output shape at every step.

In [ ]:
# Prepare input tensor: [1, C, T]
eeg_tensor = torch.from_numpy(eeg_data).float().unsqueeze(0).to(device)
T_eeg = eeg_tensor.shape[2]
eeg_lengths = torch.tensor([T_eeg], dtype=torch.long).to(device)

print("┌─ INPUT ─────────────────────────────────────────────────────┐")
print(f"│  EEG Tensor          : {list(eeg_tensor.shape)}")
print(f"│  → [batch=1, channels={eeg_tensor.shape[1]}, frames={T_eeg}]")
print(f"│  EEG Sample Rate     : {EEG_SAMPLE_RATE} Hz")
print(f"│  EEG Duration        : {T_eeg / EEG_SAMPLE_RATE:.3f} s")
print("└─────────────────────────────────────────────────────────────┘")

In [ ]:
print("╔══ EEG MODULE — Layer-by-Layer Dimension Trace ══════════════╗")

with torch.no_grad():
    x = eeg_tensor.clone()

    # 1) Conv Encoder (6 layers, stride=1)
    conv_out = eeg_module.conv_encoder(x)
    print(f"║  conv_encoder output  : {list(conv_out.shape)}")
    print(f"║    → [B, {conv_out.shape[1]}, {conv_out.shape[2]}]  (temporal preserved, stride=1)")
    print("║")

    # 2) Conv Encoder 2 (1 layer, stride=3) — temporal downsampling
    conv_out2 = eeg_module.conv_encoder2(conv_out)
    print(f"║  conv_encoder2 output : {list(conv_out2.shape)}")
    print(f"║    → [B, {conv_out2.shape[1]}, {conv_out2.shape[2]}]  (temporal down ×3)")
    print("║")

    # 3) S4 Model — sequence modeling
    if eeg_module.use_s4:
        s4_input = conv_out2.transpose(-1, -2)  # [B, T, C]
        print(f"║  S4 input (transposed): {list(s4_input.shape)}  [B, T, d_model]")
        s4_output = eeg_module.s4_model(s4_input)
        mid_output = s4_output.transpose(1, 2)  # [B, C, T]
        print(f"║  S4 output            : {list(s4_output.shape)}  [B, T, d_model]")
        print(f"║  mid_output (trans.)   : {list(mid_output.shape)}  [B, C, T]")
    else:
        mid_output = conv_out2
        print(f"║  (S4 skipped — using conv_out2 directly)")
        print(f"║  mid_output            : {list(mid_output.shape)}")
    print("║")

    # 4) Subject Discriminator — adversarial invariance
    sid_input = mid_output.mean(dim=2)  # [B, C]
    sid_logits = eeg_module.subject_discriminator(sid_input, 1.0)
    print(f"║  subject_disc input   : {list(sid_input.shape)}  (mean over time)")
    print(f"║  subject_disc output  : {list(sid_logits.shape)}  (logits for subjects)")
    print("║")

    # 5) Decoder (reconstruction branch)
    dec_out = eeg_module.deconv_encoder2(mid_output)
    print(f"║  deconv_encoder2      : {list(dec_out.shape)}  (temporal up ×3)")
    dec_out2 = eeg_module.deconv_encoder(dec_out)
    print(f"║  deconv_encoder       : {list(dec_out2.shape)}  (reconstruct EEG)")

print("╚═════════════════════════════════════════════════════════════╝")

---
## 5. Forward Pass — Tracing Dimensions Through Speech Decoder

The `mid_output` from the EEG Module is fed into the Speech Decoder in **inference mode**.

In [ ]:
print("╔══ SPEECH DECODER (inference) — Layer-by-Layer ══════════════╗")

with torch.no_grad():
    T_mid = mid_output.shape[2]
    mid_lengths = (eeg_lengths.float() * T_mid / T_eeg).long()
    print(f"║  mid_output           : {list(mid_output.shape)}")
    print(f"║  mid_output_lengths   : {mid_lengths.tolist()}")
    print("║")

    # ── 1) Connector (Transformer Encoder + projection) ──
    x_conn, m_p, logs_p, x_mask = net_g.enc_proj(mid_output, mid_lengths)
    print(f"║  ── Connector (Transformer Encoder) ──")
    print(f"║  encoder output       : {list(x_conn.shape)}")
    print(f"║  m (mean)             : {list(m_p.shape)}")
    print(f"║  logs (log-variance)  : {list(logs_p.shape)}")
    print(f"║  x_mask               : {list(x_mask.shape)}")
    print("║")

    # ── 2) Latent Sampling ──
    import commons
    y_lengths = mid_lengths.clone()
    y_mask = torch.unsqueeze(commons.sequence_mask(y_lengths, None), 1).to(x_mask.dtype)
    m_p_c = m_p[:, :, :y_mask.size(2)]
    logs_p_c = logs_p[:, :, :y_mask.size(2)]
    noise_scale = 0.667
    z_p = m_p_c + torch.randn_like(m_p_c) * torch.exp(logs_p_c) * noise_scale
    print(f"║  ── Latent Sampling ──")
    print(f"║  z_p (sampled latent) : {list(z_p.shape)}")
    print("║")

    # ── 3) Normalizing Flow (reverse) ──
    z = net_g.flow(z_p, y_mask, g=None, reverse=True)
    print(f"║  ── Normalizing Flow (reverse) ──")
    print(f"║  z (decoded latent)   : {list(z.shape)}")
    print("║")

    # ── 4) HiFi-GAN Generator — trace each upsample block ──
    max_len = 1000
    gen_input = (z * y_mask)[:, :, :max_len]
    print(f"║  ── HiFi-GAN Generator ──")
    print(f"║  generator input      : {list(gen_input.shape)}")

    g_x = net_g.dec.conv_pre(gen_input)
    print(f"║  conv_pre             : {list(g_x.shape)}")

    for i in range(net_g.dec.num_upsamples):
        g_x = torch.nn.functional.leaky_relu(g_x, 0.1)
        g_x = net_g.dec.ups[i](g_x)
        xs = None
        for j in range(net_g.dec.num_kernels):
            if xs is None:
                xs = net_g.dec.resblocks[i * net_g.dec.num_kernels + j](g_x)
            else:
                xs += net_g.dec.resblocks[i * net_g.dec.num_kernels + j](g_x)
        g_x = xs / net_g.dec.num_kernels
        rate = hps.model.upsample_rates[i]
        print(f"║  upsample[{i}] (×{rate:>2d})      : {list(g_x.shape)}")

    g_x = torch.nn.functional.leaky_relu(g_x)
    g_x = net_g.dec.conv_post(g_x)
    g_x = torch.tanh(g_x)
    print(f"║  conv_post + tanh     : {list(g_x.shape)}")

print("╚═════════════════════════════════════════════════════════════╝")

---
## 6. Full Inference & Output Audio Summary

Run the complete `SpeechDecoder.infer()` and summarize the output waveform.

In [ ]:
with torch.no_grad():
    y_hat, _, mask, *_ = net_g.infer(mid_output, mid_lengths, max_len=1000, noise_scale=0.667)

audio_sr = hps.data.sampling_rate
T_audio = y_hat.shape[2]
audio_duration = T_audio / audio_sr

print("┌─ OUTPUT AUDIO WAVEFORM ─────────────────────────────────────┐")
print(f"│  Waveform shape       : {list(y_hat.shape)}")
print(f"│  → [batch=1, channels=1, frames={T_audio}]")
print(f"│  Audio Sample Rate    : {audio_sr} Hz")
print(f"│  Audio Frames         : {T_audio}")
print(f"│  Audio Duration       : {audio_duration:.3f} seconds")
print(f"│  Frequency Range      : 0 – {audio_sr // 2} Hz (Nyquist)")
print("└─────────────────────────────────────────────────────────────┘")

---
## 7. Dimension Summary Table

In [ ]:
print("\n┌─ COMPLETE DIMENSION SUMMARY ───────────────────────────────────────────────┐")
print(f"│  {'Stage':<30s} {'Shape':<25s} {'Note':<25s} │")
print(f"│  {'─'*30} {'─'*25} {'─'*25} │")

rows = [
    ('EEG Input',              f'[1, {eeg_tensor.shape[1]}, {T_eeg}]',      f'@ {EEG_SAMPLE_RATE} Hz'),
    ('conv_encoder (6×)',      str(list(conv_out.shape)),                    'stride=1, k=4'),
    ('conv_encoder2 (1×)',     str(list(conv_out2.shape)),                   'stride=3, k=4'),
    ('S4 mid_output',          str(list(mid_output.shape)),                  '4 S4 blocks'),
    ('Connector',              str(list(x_conn.shape)),                      '6-layer Transformer'),
    ('z_p (sampled latent)',    str(list(z_p.shape)),                         '+ noise'),
    ('z (flow reverse)',        str(list(z.shape)),                           '4× ResidualCoupling'),
    ('Generator output',       str(list(y_hat.shape)),                       f'@ {audio_sr} Hz'),
]

for stage, shape, note in rows:
    print(f"│  {stage:<30s} {shape:<25s} {note:<25s} │")

print(f"└────────────────────────────────────────────────────────────────────────────┘")

print(f"\n🧠 EEG: {n_channels} channels × {n_frames} frames @ {EEG_SAMPLE_RATE} Hz = {duration_sec:.3f}s")
print(f"🔊 Audio: 1 channel × {T_audio} frames @ {audio_sr} Hz = {audio_duration:.3f}s")

---
## 8. Play Generated Audio

In [ ]:
print(f"Playing synthesized audio ({audio_duration:.2f}s) at {audio_sr} Hz...")
ipd.Audio(y_hat[0].cpu().numpy(), rate=audio_sr)

---
## ✅ Tutorial Complete!

You've traced the full forward pass from raw EEG to synthesized audio waveform, seeing every intermediate dimension along the way.